# Análise Exploratória — Acidentes de Trânsito no SUS

**TCC — Bacharelado em Sistemas de Informação, IFBA**

Notebook para exploração interativa dos dados do pipeline analítico.
Permite consultar todas as camadas (Bronze, Silver, Gold) e plotar resultados.

---

## Configuração do Ambiente

### Pré-requisitos

1. Execute o pipeline ETL antes de usar este notebook:
   ```bash
   # Dados amostrais (offline, ~2s)
   uv run python -m data-pipeline.run
   
   # OU dados reais do DATASUS (requer internet)
   uv run python -m data-pipeline.run --real --ufs BA --anos 2022 2023 2024
   ```

2. Inicie o Jupyter a partir da **raiz do projeto** (não da pasta `notebooks/`):
   ```bash
   uv run jupyter notebook
   ```

### Estrutura de dados esperada
```
data/
├── bronze/sim_parts/*.parquet   # Dados brutos SIM
├── bronze/sia_parts/*.parquet   # Dados brutos SIA
├── silver/sim.parquet           # Filtrado V01-V89, tipado
├── silver/sia.parquet           # Filtrado V01-V89, tipado
├── gold/obitos_municipio_mes.parquet
├── gold/custos_municipio_mes.parquet
├── ibge_municipios.parquet
└── ibge_populacao.parquet
```

In [ ]:
import sys
from pathlib import Path

import duckdb
import pandas as pd

# Detectar raiz do projeto (funciona tanto de notebooks/ quanto da raiz)
if Path("data").exists():
    PROJECT_ROOT = Path.cwd()
elif Path("../data").exists():
    PROJECT_ROOT = Path.cwd().parent
else:
    raise FileNotFoundError("Execute o pipeline ETL antes: uv run python -m data-pipeline.run")

# Adicionar raiz ao path para importar módulos do projeto
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

# Conexão DuckDB com views Gold (mesma estrutura do backend)
con = duckdb.connect(":memory:")

# Registrar views Gold
con.sql(f"CREATE VIEW v_obitos AS SELECT * FROM read_parquet('{GOLD_DIR}/obitos_municipio_mes.parquet')")
con.sql(f"CREATE VIEW v_custos AS SELECT * FROM read_parquet('{GOLD_DIR}/custos_municipio_mes.parquet')")

# Views IBGE (se existirem)
ibge_mun = DATA_DIR / "ibge_municipios.parquet"
ibge_pop = DATA_DIR / "ibge_populacao.parquet"
if ibge_mun.exists():
    con.sql(f"CREATE VIEW v_ibge_municipios AS SELECT * FROM read_parquet('{ibge_mun}')")
if ibge_pop.exists():
    con.sql(f"CREATE VIEW v_ibge_populacao AS SELECT * FROM read_parquet('{ibge_pop}')")

print(f"Projeto: {PROJECT_ROOT}")
print(f"DuckDB {duckdb.__version__} | Pandas {pd.__version__}")
print(f"Views registradas: {[r[0] for r in con.sql('SHOW TABLES').fetchall()]}")

---

## Consulta Rápida (use esta célula para explorar livremente)

Troque a query SQL abaixo para consultar qualquer dado das views Gold.

**Views disponíveis:**
- `v_obitos` — óbitos agregados por município/mês (colunas: `cod_mun_ibge`, `municipio`, `uf`, `competencia`, `ano`, `mes`, `total_obitos`, `tipo_veiculo`, `faixa_etaria`, `sexo`, `lat`, `lon`, `populacao_estimada`)
- `v_custos` — custos ambulatoriais por município/mês (colunas: `cod_mun_ibge`, `municipio`, `uf`, `competencia`, `ano`, `mes`, `custo_total`, `total_procedimentos`, `total_atendimentos`, `tipo_veiculo`, `faixa_etaria`, `lat`, `lon`)
- `v_ibge_municipios` — nome, UF, região, lat, lon
- `v_ibge_populacao` — população estimada por município/ano

In [ ]:
# === CONSULTA LIVRE ===
# Troque a query abaixo e execute (Shift+Enter)

con.sql("""
    SELECT municipio, ano, SUM(total_obitos) AS obitos
    FROM v_obitos
    GROUP BY municipio, ano
    ORDER BY obitos DESC
    LIMIT 20
""").fetchdf()

---

## 1. Visão Geral dos Dados

In [ ]:
# Resumo geral
resumo = con.sql("""
    SELECT
        (SELECT SUM(total_obitos) FROM v_obitos) AS total_obitos,
        (SELECT ROUND(SUM(custo_total), 2) FROM v_custos) AS custo_total_r$,
        (SELECT SUM(total_atendimentos) FROM v_custos) AS total_atendimentos,
        (SELECT COUNT(DISTINCT cod_mun_ibge) FROM v_obitos) AS municipios_obitos,
        (SELECT COUNT(DISTINCT cod_mun_ibge) FROM v_custos) AS municipios_custos,
        (SELECT MIN(ano) FROM v_obitos) AS ano_inicio,
        (SELECT MAX(ano) FROM v_obitos) AS ano_fim
""").fetchdf()

print("="*60)
print("  RESUMO GERAL DO PIPELINE")
print("="*60)
r = resumo.iloc[0]
print(f"  Óbitos:        {int(r['total_obitos']):>10,}")
print(f"  Custos SUS:    R$ {r['custo_total_r$']:>13,.2f}")
print(f"  Atendimentos:  {int(r['total_atendimentos']):>10,}")
print(f"  Municípios:    {int(r['municipios_obitos']):>10} (óbitos) / {int(r['municipios_custos'])} (custos)")
print(f"  Período:       {int(r['ano_inicio'])} a {int(r['ano_fim'])}")
print("="*60)

In [ ]:
# Schema detalhado das views Gold
print("=== v_obitos ===")
con.sql("DESCRIBE v_obitos").fetchdf()

In [ ]:
print("=== v_custos ===")
con.sql("DESCRIBE v_custos").fetchdf()

---

## 2. Análise Temporal

In [ ]:
# Óbitos e custos por ano
temporal = con.sql("""
    SELECT
        o.ano,
        o.total_obitos,
        c.custo_total,
        c.atendimentos
    FROM (
        SELECT ano, SUM(total_obitos) AS total_obitos FROM v_obitos GROUP BY ano
    ) o
    LEFT JOIN (
        SELECT ano, ROUND(SUM(custo_total),2) AS custo_total,
               SUM(total_atendimentos) AS atendimentos
        FROM v_custos GROUP BY ano
    ) c ON o.ano = c.ano
    ORDER BY o.ano
""").fetchdf()

temporal

In [ ]:
# Gráfico: evolução anual de óbitos
ax = temporal.plot(
    x="ano", y="total_obitos",
    kind="bar", figsize=(10, 4), color="#ef4444",
    title="Óbitos por Acidentes de Trânsito por Ano",
    legend=False
)
ax.set_xlabel("Ano")
ax.set_ylabel("Óbitos")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width()/2., p.get_height()),
                ha='center', va='bottom', fontsize=9)

In [ ]:
# Série temporal mensal
serie = con.sql("""
    SELECT STRFTIME(competencia, '%Y-%m') AS mes, SUM(total_obitos) AS obitos
    FROM v_obitos
    GROUP BY STRFTIME(competencia, '%Y-%m')
    ORDER BY mes
""").fetchdf()

ax = serie.plot(x="mes", y="obitos", figsize=(14, 4), color="#3b82f6",
                title="Série Temporal Mensal — Óbitos", legend=False)
ax.set_xlabel("")
ax.set_ylabel("Óbitos/mês")
ax.tick_params(axis='x', rotation=45, labelsize=7)

In [ ]:
# Sazonalidade mensal (média)
sazon = con.sql("""
    SELECT mes, ROUND(AVG(total), 1) AS media_obitos
    FROM (
        SELECT mes, SUM(total_obitos) AS total
        FROM v_obitos GROUP BY ano, mes
    ) GROUP BY mes ORDER BY mes
""").fetchdf()

ax = sazon.plot(x="mes", y="media_obitos", kind="bar", figsize=(10, 4),
                color="#f59e0b", title="Sazonalidade — Média de Óbitos por Mês", legend=False)
ax.set_xlabel("Mês")
ax.set_ylabel("Média de óbitos")

---

## 3. Distribuições

In [ ]:
# Por tipo de veículo
tipo = con.sql("""
    SELECT tipo_veiculo, SUM(total_obitos) AS total,
           ROUND(SUM(total_obitos)*100.0 / (SELECT SUM(total_obitos) FROM v_obitos), 1) AS pct
    FROM v_obitos GROUP BY tipo_veiculo ORDER BY total DESC
""").fetchdf()

ax = tipo.plot(x="tipo_veiculo", y="total", kind="barh", figsize=(10, 5),
               color="#8b5cf6", title="Óbitos por Tipo de Veículo", legend=False)
ax.set_ylabel("")
ax.invert_yaxis()
for i, (_, row) in enumerate(tipo.iterrows()):
    ax.text(row['total'] + 10, i, f"{int(row['total']):,} ({row['pct']}%)", va='center', fontsize=9)

tipo

In [ ]:
# Por faixa etária
faixa = con.sql("""
    SELECT faixa_etaria, SUM(total_obitos) AS total,
           ROUND(SUM(total_obitos)*100.0 / (SELECT SUM(total_obitos) FROM v_obitos), 1) AS pct
    FROM v_obitos GROUP BY faixa_etaria
    ORDER BY CASE faixa_etaria
        WHEN '0-14' THEN 1 WHEN '15-24' THEN 2 WHEN '25-34' THEN 3
        WHEN '35-44' THEN 4 WHEN '45-54' THEN 5 WHEN '55-64' THEN 6 ELSE 7 END
""").fetchdf()

ax = faixa.plot(x="faixa_etaria", y="total", kind="bar", figsize=(10, 4),
                color=["#3b82f6","#ef4444","#f59e0b","#10b981","#8b5cf6","#ec4899","#06b6d4"],
                title="Óbitos por Faixa Etária", legend=False)
ax.set_xlabel("")
ax.set_ylabel("Óbitos")

faixa

In [ ]:
# Por sexo
sexo = con.sql("""
    SELECT sexo, SUM(total_obitos) AS total,
           ROUND(SUM(total_obitos)*100.0 / (SELECT SUM(total_obitos) FROM v_obitos), 1) AS pct
    FROM v_obitos GROUP BY sexo ORDER BY total DESC
""").fetchdf()

sexo.plot(x="sexo", y="total", kind="pie", figsize=(6, 6),
          autopct='%1.1f%%', colors=["#3b82f6", "#ec4899"],
          title="Óbitos por Sexo", legend=False, ylabel="")

sexo

---

## 4. Ranking de Municípios

In [ ]:
# Top 15 municípios por óbitos
top_mun = con.sql("""
    SELECT municipio, uf, SUM(total_obitos) AS obitos
    FROM v_obitos
    GROUP BY municipio, uf
    ORDER BY obitos DESC
    LIMIT 15
""").fetchdf()

ax = top_mun.plot(x="municipio", y="obitos", kind="barh", figsize=(10, 6),
                  color="#ef4444", title="Top 15 Municípios — Óbitos", legend=False)
ax.set_ylabel("")
ax.invert_yaxis()

top_mun

In [ ]:
# Top 15 municípios por custos
top_custos = con.sql("""
    SELECT municipio, uf,
           ROUND(SUM(custo_total), 2) AS custo_r$,
           SUM(total_atendimentos) AS atendimentos
    FROM v_custos
    GROUP BY municipio, uf
    ORDER BY custo_r$ DESC
    LIMIT 15
""").fetchdf()

ax = top_custos.plot(x="municipio", y="custo_r$", kind="barh", figsize=(10, 6),
                     color="#f59e0b", title="Top 15 Municípios — Custos Ambulatoriais", legend=False)
ax.set_ylabel("")
ax.invert_yaxis()

top_custos

---

## 5. Análise Detalhada por Município

Troque o `MUNICIPIO` abaixo para explorar qualquer cidade.

In [ ]:
MUNICIPIO = "Salvador"  # <-- troque aqui

detalhe = con.sql(f"""
    SELECT STRFTIME(competencia, '%Y-%m') AS mes, SUM(total_obitos) AS obitos
    FROM v_obitos
    WHERE municipio ILIKE '%{MUNICIPIO}%'
    GROUP BY STRFTIME(competencia, '%Y-%m')
    ORDER BY mes
""").fetchdf()

ax = detalhe.plot(x="mes", y="obitos", figsize=(14, 4), color="#ef4444",
                  title=f"Série Temporal — Óbitos em {MUNICIPIO}", legend=False)
ax.tick_params(axis='x', rotation=45, labelsize=7)
ax.set_xlabel("")
ax.set_ylabel("Óbitos/mês")

print(f"Total: {detalhe['obitos'].sum():,} óbitos em {len(detalhe)} meses")
detalhe

In [ ]:
# Custos do município
custos_mun = con.sql(f"""
    SELECT STRFTIME(competencia, '%Y-%m') AS mes,
           ROUND(SUM(custo_total), 2) AS custo_r$,
           SUM(total_atendimentos) AS atendimentos
    FROM v_custos
    WHERE municipio ILIKE '%{MUNICIPIO}%'
    GROUP BY STRFTIME(competencia, '%Y-%m')
    ORDER BY mes
""").fetchdf()

ax = custos_mun.plot(x="mes", y="custo_r$", figsize=(14, 4), color="#f59e0b",
                     title=f"Série Temporal — Custos em {MUNICIPIO}", legend=False)
ax.tick_params(axis='x', rotation=45, labelsize=7)
ax.set_xlabel("")
ax.set_ylabel("Custo (R$)")

print(f"Total: R$ {custos_mun['custo_r$'].sum():,.2f} em {custos_mun['atendimentos'].sum():,} atendimentos")
custos_mun

---

## 6. Inspeção das Camadas Bronze e Silver

Para navegar pelas colunas dos dados brutos (antes das transformações).

In [ ]:
# Bronze SIM — schema e amostra
sim_source = BRONZE_DIR / "sim_parts"
if not sim_source.exists():
    sim_source = BRONZE_DIR / "sim.parquet"

glob = f"{sim_source}/*.parquet" if sim_source.is_dir() else str(sim_source)
df = con.sql(f"SELECT * FROM read_parquet('{glob}') LIMIT 10").fetchdf()

print(f"Bronze SIM: {con.sql(f\"SELECT COUNT(*) FROM read_parquet('{glob}')\").fetchone()[0]:,} registros")
print(f"Colunas ({len(df.columns)}): {list(df.columns)}")
df

In [ ]:
# Bronze SIA — schema e amostra
sia_source = BRONZE_DIR / "sia_parts"
if not sia_source.exists():
    sia_source = BRONZE_DIR / "sia.parquet"

glob_sia = f"{sia_source}/*.parquet" if sia_source.is_dir() else str(sia_source)
df_sia = con.sql(f"SELECT * FROM read_parquet('{glob_sia}') LIMIT 10").fetchdf()

print(f"Bronze SIA: {con.sql(f\"SELECT COUNT(*) FROM read_parquet('{glob_sia}')\").fetchone()[0]:,} registros")
print(f"Colunas ({len(df_sia.columns)}): {list(df_sia.columns)}")
df_sia

In [ ]:
# Silver SIM — após filtro CID V01-V89 e enriquecimento
silver_sim = SILVER_DIR / "sim.parquet"
if silver_sim.exists():
    df_ss = con.sql(f"SELECT * FROM read_parquet('{silver_sim}') LIMIT 10").fetchdf()
    print(f"Silver SIM: {con.sql(f\"SELECT COUNT(*) FROM read_parquet('{silver_sim}')\").fetchone()[0]:,} registros")
    display(df_ss)
else:
    print("Silver SIM não encontrado. Execute o pipeline ETL.")

In [ ]:
# Silver SIA — após filtro CID V01-V89
silver_sia = SILVER_DIR / "sia.parquet"
if silver_sia.exists():
    df_ss_sia = con.sql(f"SELECT * FROM read_parquet('{silver_sia}') LIMIT 10").fetchdf()
    print(f"Silver SIA: {con.sql(f\"SELECT COUNT(*) FROM read_parquet('{silver_sia}')\").fetchone()[0]:,} registros")
    display(df_ss_sia)
else:
    print("Silver SIA não encontrado. Execute o pipeline ETL.")

---

## 7. Dados IBGE (População e Coordenadas)

In [ ]:
# Municípios IBGE com coordenadas
if ibge_mun.exists():
    ibge_df = con.sql("SELECT * FROM v_ibge_municipios ORDER BY nome").fetchdf()
    print(f"IBGE Municípios: {len(ibge_df)} registros")
    display(ibge_df)
else:
    print("ibge_municipios.parquet não encontrado.")

In [ ]:
# População estimada IBGE
if ibge_pop.exists():
    pop_df = con.sql("""
        SELECT p.cod_mun_ibge, m.nome, p.ano, p.populacao
        FROM v_ibge_populacao p
        LEFT JOIN v_ibge_municipios m ON LEFT(p.cod_mun_ibge, 6) = LEFT(m.cod_mun_ibge, 6)
        ORDER BY p.populacao DESC
    """).fetchdf()
    print(f"IBGE População: {len(pop_df)} registros")
    display(pop_df.head(20))
else:
    print("ibge_populacao.parquet não encontrado.")

---

## 8. Consultas SQL de Exemplo

Exemplos de consultas que o MCP Server e a API REST usam.

In [ ]:
# Exemplo: taxa de mortalidade por 100 mil habitantes
if ibge_pop.exists():
    taxa = con.sql("""
        SELECT
            o.municipio,
            o.uf,
            SUM(o.total_obitos) AS obitos,
            MAX(p.populacao) AS populacao,
            ROUND(SUM(o.total_obitos) * 100000.0 / MAX(p.populacao), 2) AS taxa_100mil
        FROM v_obitos o
        LEFT JOIN v_ibge_populacao p
            ON LEFT(o.cod_mun_ibge, 6) = LEFT(p.cod_mun_ibge, 6)
            AND o.ano = p.ano
        WHERE p.populacao > 0
        GROUP BY o.municipio, o.uf
        ORDER BY taxa_100mil DESC
        LIMIT 15
    """).fetchdf()
    print("Taxa de Mortalidade por 100 mil habitantes:")
    display(taxa)
else:
    print("Sem dados de população. Execute o pipeline com --real para gerar.")

In [ ]:
# Exemplo: custo por tipo de veículo em um município
CIDADE = "Feira de Santana"  # <-- troque aqui

custo_tipo = con.sql(f"""
    SELECT tipo_veiculo,
           ROUND(SUM(custo_total), 2) AS custo_r$,
           SUM(total_atendimentos) AS atendimentos
    FROM v_custos
    WHERE municipio ILIKE '%{CIDADE}%'
    GROUP BY tipo_veiculo
    ORDER BY custo_r$ DESC
""").fetchdf()

print(f"Custos por tipo de veículo em {CIDADE}:")
custo_tipo

---

## 9. Limpeza

Feche a conexão ao terminar.

In [ ]:
con.close()
print("Conexão DuckDB fechada.")